# Setup

In [1]:
import os
from getpass import getpass

# This will pop up an input box at the top of VS Code when you run the cell
os.environ["GROQ_API_KEY"] = getpass("Enter your GROQ_API_KEY: ")

print("Key loaded successfully!")

Enter your GROQ_API_KEY: ··········
Key loaded successfully!


In [2]:
%pip install langchain langchain-groq langchain-community langchain-text-splitters pymupdf pydantic rich arxiv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 29.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 70.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 56.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 5.6 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.


# Working with documents

A **Document Loader** is a class that:

1. Fetches content from some source (Arxiv, a local PDF, a website, …)

2. Returns a Python list of LangChain `Document` objects

Every loader returns `list[Document]`. Each `Document` contains:

1. `document.page_content`

2. `document.meta_data`

### Arxiv - Load Research papers

In [3]:
import arxiv
import urllib.request
import fitz
from langchain_core.documents import Document

def load_arxiv_papers(query:str, load_max_docs: int = 1) -> list[Document]:
    client = arxiv.Client()
    search = arxiv.Search(query=query, max_results=load_max_docs)
    documents = []

    for result in client.results(search):
        # Download the PDF to a temp file
        pdf_path = f"/tmp/{result.entry_id.split('/')[-1]}.pdf"
        urllib.request.urlretrieve(result.pdf_url, pdf_path)

        # Extract full text with PyMuPDF
        full_text = ""
        with fitz.open(pdf_path) as pdf:
            for page in pdf:
                full_text += page.get_text()

        documents.append(Document(
            page_content=full_text,
            metadata={
                "title":     result.title,
                "authors":   ", ".join(str(a) for a in result.authors),
                "published": str(result.published.date()),
                "summary":   result.summary,
                "entry_id":  result.entry_id,
                "pdf_url":   result.pdf_url,
            }
        ))

    return documents

documents = load_arxiv_papers(query="RAG", load_max_docs=2)
print(documents[0].page_content, "\n")

MultiHop-RAG: Benchmarking Retrieval-Augmented Generation for
Multi-Hop Queries
Yixuan Tang and Yi Yang
Hong Kong University of Science and Technology
{yixuantang,imyiyang}@ust.hk
Abstract
Retrieval-augmented generation (RAG) aug-
ments large language models (LLM) by re-
trieving relevant knowledge, showing promis-
ing potential in mitigating LLM hallucinations
and enhancing response quality, thereby facil-
itating the great adoption of LLMs in prac-
tice. However, we find that existing RAG sys-
tems are inadequate in answering multi-hop
queries, which require retrieving and reasoning
over multiple pieces of supporting evidence.
Furthermore, to our knowledge, no existing
RAG benchmarking dataset focuses on multi-
hop queries. In this paper, we develop a novel
dataset, MultiHop-RAG, which consists of a
knowledge base, a large collection of multi-
hop queries, their ground-truth answers, and
the associated supporting evidence. We detail
the procedure of building the dataset, utiliz-
ing 

### Local File Loaders

In [ ]:
%pip install "unstructured[pdf]" --break-system-packages

#### UnstructuredFileLoader
How it parses:

It attempts to reconstruct the structural layout of the file. It partitions the document into individual semantic blocks (e.g. distinguishing a "Title" from a "Paragraph").

Modes:

1. mode="single" (default): Combines all extracted text into one large Document.

2. mode="elements": Returns a list of separate Document objects, where each object is a specific element (like a separate table, title, list item, or paragraph).

In [4]:
from ipywidgets import FileUpload
from IPython.display import display

upload = FileUpload(accept='', multiple=False)
display(upload)

FileUpload(value={}, description='Upload')

In [5]:
filename = list(upload.value.keys())[0]
with open(filename, "wb") as f:
    f.write(upload.value[filename]["content"])

# you can pass file name to any loader
print(upload)

FileUpload(value={'main.pdf': {'metadata': {'name': 'main.pdf', 'type': 'application/pdf', 'size': 130674, 'lastModified': 1779992082706}, 'content': b'%PDF-1.7\n%\xd0\xd4\xc5\xd8\n37 0 obj\n<<\n/Length 3397      \n/Filter /FlateDecode\n>>\nstream\nx\xda\xbdZ\xdbr\xdbF\x12}\xf7W\xb0j\xcb)\xb0,\xc2\xb8\x03\xcc\xbe,MY\xb6\\\x92\xcd\x98t\xb2\xb5N\x1eF\xe0\x88\x84\x85\x0b\x03\x80\xb2\xb9\xce\xc7o\xf7t\x0f\x00J -\xbb\xa2}\x11Ab.=}9\xa7\xbbG\xd6`5\xb0\x06\xaf\x9eX\x07>_,\x9e<?\xf3\xdc\x81\xe3\x99a\x14\xda\x83\xc5\xf5\xc0qls\xec\x07\x83\xd0wM;\n\x07\x8b\xe5\xe0\xa31)\xd7"\x1b\x8e\xdc\xd07\xce\xb3R\xe4\xc3?\x16o`\xa27\xb0q\xb0\xef\xe0D\xcf2\x03\xcb\x1b\x8c\xec\xc0\xf4\xdd\x88\xe6]\x88uQ\xca\x13\x98\xe9\x04\xc6l\xe8\x84\x86\xb8I\xaaZ/\xe0:\x83\xb19\x0e\x9c\x00\xe7\x8fl74\xfd1~\x9ac\x8f\x17x6v`\xae7\x82=l\xdf\x8axZ\xd4\x99\xf6\x11\xc6\xbb\xae\xf1\xa9gE\xf5\xca3\x12\x94X\xe0\t\xc2q\xf4/\x99\x89$5\xe3"\xfb\x81\xb5\\c\x95\xd4k<\xc6\xf6\xca\x1c\xdaF<\x1c\xd9F\x91=OD\xa64d[8s\xe0\xb9f\x14\xb8\xfb\x07

In [7]:
from langchain_community.document_loaders import PyMuPDFLoader

loader = PyMuPDFLoader("main.pdf")
docs = loader.load()

print(len(docs))
print(docs[0].page_content[:500])

1
Arham Imran
Lahore, Pakistan
+92-324-5521508
|
imranarham798@email.com
|
github.com/iamArham10
linkedin.com/in/iarham-imran
|
iamarham10.github.io/me
Summary
Final-year CS student at UET Lahore with hands-on experience building full-stack applications using React, Django, and
FastAPI. Passionate about AI/ML and network security, seeking a software engineering internship to apply strong
development and problem-solving skills.
Education
University of Engineering & Technology
Lahore, Pakistan
Bache


## Chunking

A typical research paper has 50,000 - 150,000 Characters, far too long to fit in a single LLM Call. We split it into smaller chunks that fit comfortably in context window

In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=600, # Max char per chunk
    chunk_overlap=100, # How many char next chunk uses from previous for overlap.
    separators=["\n\n", "\n", ".", ";", ",", " ", ""], # Separators to try in order
    length_function=len
)

splits = text_splitter.split_documents(docs)

In [9]:
print(f"Number Of Chunks: {len(splits)}")
print(f"Avg Chunk Lenght: {sum(len(d.page_content) for d in splits) // len(splits)} chars")
for i in [0, 1, 2, -1]:
    print(f"\n[Chunk {i}] ({len(splits[i].page_content)} chars)")
    print(splits[i].page_content)
    print("─" * 60)

Number Of Chunks: 7
Avg Chunk Lenght: 526 chars

[Chunk 0] (589 chars)
Arham Imran
Lahore, Pakistan
+92-324-5521508
|
imranarham798@email.com
|
github.com/iamArham10
linkedin.com/in/iarham-imran
|
iamarham10.github.io/me
Summary
Final-year CS student at UET Lahore with hands-on experience building full-stack applications using React, Django, and
FastAPI. Passionate about AI/ML and network security, seeking a software engineering internship to apply strong
development and problem-solving skills.
Education
University of Engineering & Technology
Lahore, Pakistan
Bachelor of Science in Computer Science; GPA: 3.47/4.0
Sep 2021 – Jun 2026
Relevant Coursework
────────────────────────────────────────────────────────────

[Chunk 1] (586 chars)
Bachelor of Science in Computer Science; GPA: 3.47/4.0
Sep 2021 – Jun 2026
Relevant Coursework
Object Oriented Programming, Data Structures & Algorithms, Operating Systems, DBMS, Computer Networks,
Software Engineering, Artificial Intelligence, Machine Le

Token based chunking:

The split above counts characters. But LLM's have limits measured in tokens(roughly 1 token = 0.75 words). For more accurate splitting

In [10]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name="cl100k_base",
    chunk_size=300,
    chunk_overlap=30,
)

In [11]:
splits = text_splitter.split_documents(docs)
print(f"Number Of Chunks: {len(splits)}")
print(f"Avg Chunk Lenght: {sum(len(d.page_content) for d in splits) // len(splits)} chars")
for i in [0, 1, 2, -1]:
    print(f"\n[Chunk {i}] ({len(splits[i].page_content)} chars)")
    print(splits[i].page_content)
    print("─" * 60)

Number Of Chunks: 3
Avg Chunk Lenght: 1161 chars

[Chunk 0] (1247 chars)
Arham Imran
Lahore, Pakistan
+92-324-5521508
|
imranarham798@email.com
|
github.com/iamArham10
linkedin.com/in/iarham-imran
|
iamarham10.github.io/me
Summary
Final-year CS student at UET Lahore with hands-on experience building full-stack applications using React, Django, and
FastAPI. Passionate about AI/ML and network security, seeking a software engineering internship to apply strong
development and problem-solving skills.
Education
University of Engineering & Technology
Lahore, Pakistan
Bachelor of Science in Computer Science; GPA: 3.47/4.0
Sep 2021 – Jun 2026
Relevant Coursework
Object Oriented Programming, Data Structures & Algorithms, Operating Systems, DBMS, Computer Networks,
Software Engineering, Artificial Intelligence, Machine Learning, Natural Language Processing, Computer Vision
Projects
Agentified Marketing Automation Platform | React.js, Django, LangChain, Celery, Redis
2025 – Present
• Built an AI-

### Pydantic Models

We want LLMs to return structured data, pydantic models make it easy for us

In [12]:
from pydantic import BaseModel, Field
from typing import List

class DocumentSummaryBase(BaseModel):
    running_summary: str = Field(
        "",  # default value

        # description (llm read this)
        description="Running description of the document. Do not override; only update!"
    )
    main_ideas: List[str] = Field(
        [],
        description="Most important information from the document (max 3)"
    )
    loose_ends: List[str] = Field(
        [],
        description="Open questions to incorporate into future summaries (max 3)"
    )

In [13]:
from langchain_core.output_parsers import PydanticOutputParser

parser = PydanticOutputParser(pydantic_object=DocumentSummaryBase)

instructions = parser.get_format_instructions()

# parser
# parsed_object = parser.parse(llm_output_string)

In [14]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template(
    "Summarize the following text: {input}\n"
    "Current summary so far: {info_base}\n"
    "{format_instructions}"
)

RExtract function bundles a prompt + llm + pydantic parser together

In [18]:
from langchain_core.runnables import RunnableAssign
from langchain_core.output_parsers import PydanticOutputParser

def RExtract(pydantic_class, llm, parser):
    parser = PydanticOutputParser(pydantic_object=pydantic_class)

    instruct_merge = RunnableAssign(
        {"format_instructions": lambda x: parser.get_format_instructions()}
    )

    def preparse(string: str) -> str:
        """Fix the most common LLM JSON formatting mistakes."""
        if "{" not in string: string = "{" + string
        if "}" not in string: string = string + "}"
        string = (string
            .replace("\\_", "_")
            .replace("\\n", " ")
            .replace("\\]", "]")
            .replace("\\[", "[")
        )
        return string

    return instruct_merge | prompt | llm | preparse | parser.parse

### Document Refinement (RSummarizer)

Instead of trying to summarize all chunks at once, process one chunk at a time and refine knowledge base

In [21]:
from langchain_core.prompts import ChatPromptTemplate

summary_prompt = ChatPromptTemplate.from_template(
    "You are generating a running summary of the document. Make it readable by a technical user."
    " After this, the old knowledge base will be replaced by the new one."
    " Keep it short but dense and useful."
    " Information should flow: chunk → (loose_ends / main_ideas) → running_summary."
    " Current knowledge base: {info_base}."
    "\n\n{format_instructions}. Follow the format precisely, including quotations and commas."
    "\n\nWithout losing any existing info, update the knowledge base with: {input}"
)

In [28]:
from langchain_core.runnables import RunnableLambda
from IPython.display import clear_output

def RSummarizer(knowledge_class, llm, prompt, verbose=False):
    parse_chain = RExtract(knowledge_class, llm, prompt)

    def summarize_docs(docs):
        # Start with an empty schema instance
        state = {"info_base": knowledge_class()}

        for i, doc in enumerate(docs):
            state = {
                "input"    : doc.page_content,
                "info_base": state["info_base"],
            }
            new_summary  = parse_chain.invoke(state)
            state["info_base"] = new_summary

            if verbose:
                print(f"Processed chunk {i+1}/{len(docs)}")
                print(new_summary)

        return state["info_base"]

    return RunnableLambda(summarize_docs)

In [29]:
from langchain_groq import ChatGroq
from langchain_core.output_parsers import StrOutputParser
instruct_model = ChatGroq(model="llama-3.3-70b-versatile", temperature=0, max_tokens=2048)
instruct_llm   = instruct_model | StrOutputParser()

summarizer = RSummarizer(
    knowledge_class=DocumentSummaryBase,
    llm=instruct_llm,
    prompt=summary_prompt,
    verbose=True,
)

# summarize the first 10 chunks for example
final_summary = summarizer.invoke(splits[:10])

print("Running summary:")
print(final_summary.running_summary)

print("\nMain ideas:")
for idea in final_summary.main_ideas:
    print(f"  • {idea}")

print("\nLoose ends / open questions:")
for end in final_summary.loose_ends:
    print(f"  • {end}")

Processed chunk 1/3
running_summary='Arham Imran is a final-year CS student at UET Lahore with experience in building full-stack applications and a passion for AI/ML and network security. He is seeking a software engineering internship to apply his development and problem-solving skills.' main_ideas=['Arham Imran is a final-year CS student at UET Lahore', 'He has experience in building full-stack applications using React, Django, and FastAPI', 'He is passionate about AI/ML and network security'] loose_ends=['What specific skills is Arham looking to apply in his internship?', 'How does his experience with Agentified Marketing Automation Platform relate to his career goals?', 'What are his plans after completing his Bachelor of Science in Computer Science?']
Processed chunk 2/3
running_summary='Arham Imran is a final-year CS student at UET Lahore with experience in building full-stack applications and a passion for AI/ML and network security. He has worked on projects such as brand voice

In [32]:
from pprint import pprint as pprint
pprint(final_summary.dict())

{'loose_ends': ['What specific skills is Arham looking to apply in his '
                'internship?',
                'How does his experience with AI/ML and network security '
                'relate to his career goals?',
                'What are his plans after completing his Bachelor of Science '
                'in Computer Science?'],
 'main_ideas': ['Arham Imran is a final-year CS student at UET Lahore with '
                'experience in full-stack development and AI/ML',
                'He has worked on multiple projects, including image '
                'classification and network traffic analysis',
                'He is seeking a software engineering internship to apply his '
                'skills and has a strong technical foundation'],
 'running_summary': 'Arham Imran is a final-year CS student at UET Lahore with '
                    'experience in building full-stack applications and a '
                    'passion for AI/ML and network security. He has worked 

/tmp/ipykernel_6622/1114954133.py:2: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  pprint(final_summary.dict())
